In [1]:
import torch
import torch.nn as nn
import random
import time
import cv2
import numpy as np
import keyboard
import pyautogui
from collections import deque
from IPython.display import clear_output

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from hand_visualizer import HandVisualizer

In [2]:
# ── Input ─────────────────────────────────────────────────────────────────────
num_hands       = 1
tracking_points = 21
tracking_dim    = 3

# ── Autoencoder ───────────────────────────────────────────────────────────────
latent_dim  = 16    # bottleneck size — independent of n_regions
hidden_dim  = 256   # hidden layer width

# ── Region detector ───────────────────────────────────────────────────────────
n_regions       = 4
action_labels   = ['a', 's', 'd', 'w']
centroid_mom    = 0.98   # EMA momentum; lower = faster drift tracking

# ── Online training ───────────────────────────────────────────────────────────
ae_lr           = 1e-4
replay_size     = 256   # max frames in the replay buffer
replay_batch    = 32    # mini-batch size per AE gradient step

# ── Warmup (run once before going live) ───────────────────────────────────────
warmup_frames   = 300   # frames to collect before online mode
warmup_epochs   = 80    # AE batch-train epochs on warmup data

# ── Misc ──────────────────────────────────────────────────────────────────────
run_device    = torch.device('cuda')
press_buttons = False

In [3]:
class CameraWrapper:
    def __init__(self, camera_index: int = 0):
        self.cap = cv2.VideoCapture(camera_index)
        if not self.cap.isOpened():
            raise IOError(f'Cannot open camera at index {camera_index}')

    def __call__(self) -> np.ndarray | None:
        ret, frame = self.cap.read()
        return frame if ret else None

    def release(self):
        if self.cap.isOpened():
            self.cap.release()

    def __enter__(self):  return self
    def __exit__(self, *_): self.release()

In [4]:
class HandTracker:
    def __init__(self, model_path, max_num_hands=2,
                 min_hand_detection_confidence=0.5,
                 min_hand_presence_confidence=0.5,
                 min_tracking_confidence=0.5):
        base = python.BaseOptions(model_asset_path=model_path)
        self.options = vision.HandLandmarkerOptions(
            base_options=base,
            running_mode=vision.RunningMode.VIDEO,
            num_hands=max_num_hands,
            min_hand_detection_confidence=min_hand_detection_confidence,
            min_hand_presence_confidence=min_hand_presence_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)

    def reset(self):
        self.landmarker.close()
        self.landmarker = vision.HandLandmarker.create_from_options(self.options)

    def __call__(self, bgr_frame: np.ndarray, timestamp_ms: int) -> torch.Tensor:
        rgb    = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        result = self.landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
            timestamp_ms=timestamp_ms,
        )
        if not result.hand_landmarks:
            return torch.empty((0, 21, 3), dtype=torch.float32)
        out = np.zeros((len(result.hand_landmarks), 21, 3), dtype=np.float32)
        for h, hand in enumerate(result.hand_landmarks):
            for i, lm in enumerate(hand):
                out[h, i] = [lm.x, lm.y, lm.z]
        return torch.from_numpy(out)

    def close(self):
        self.landmarker.close()

In [5]:
_TOPOLOGY = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]

def to_bone_offsets(tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert absolute MediaPipe landmark coords (N, 21, 3) to
    parent-relative bone offsets. Wrist becomes (0,0,0); each joint
    is its displacement from its parent. Translation-invariant.
    """
    out = tensor.clone()
    for h in range(out.shape[0]):
        out[h, 0] = 0.0
        for parent, child in _TOPOLOGY:
            out[h, child] = tensor[h, child] - tensor[h, parent]
    return out

In [6]:
class Encoder(nn.Module):
    """
    Maps bone-offset hand pose (N, num_hands, 21, 3) → latent z (N, latent_dim).

    Architecture:
        Linear → ELU → Linear → ELU → Linear → Tanh

    Tanh at the output bounds every latent dimension to (−1, 1), which keeps
    the region-detector's k-means well-conditioned as the encoder drifts.
    """
    def __init__(self):
        super().__init__()
        input_dim = num_hands * tracking_points * tracking_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.net(x.flatten(start_dim=1))


class Decoder(nn.Module):
    """
    Maps latent z (N, latent_dim) → reconstructed bone offsets (N, input_dim).

    No final activation — bone offsets are unconstrained real-valued differences.
    """
    def __init__(self):
        super().__init__()
        output_dim = num_hands * tracking_points * tracking_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class PoseAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        z     = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

In [7]:
class RegionDetector:
    """
    Online k-means region detector operating in the AE latent space.

    Seeding (once, on warmup data)
    ───────────────────────────────
    K-Means++ selects k initial centroids from a batch of latent vectors
    so they are well-spread. This avoids the collapsed-cluster problem
    that random init causes in low-density regions.

    Per-frame operation
    ───────────────────
    query_and_update(z):
        1. Find the nearest centroid by squared Euclidean distance.
        2. Pull that centroid toward z via EMA:
               centroid[k] ← mom * centroid[k] + (1−mom) * z
           This lets centroids track the latent distribution as the
           encoder drifts during online training.
        3. Return the winning region index.

    All tensors stay on CPU; z is expected detached.
    """

    def __init__(self, n_regions: int, latent_dim: int, momentum: float = 0.98):
        self.n_regions  = n_regions
        self.latent_dim = latent_dim
        self.momentum   = momentum
        self.centroids  = None   # (n_regions, latent_dim) — set by seed_from_batch

    # ------------------------------------------------------------------

    def seed_from_batch(self, z_batch: torch.Tensor) -> None:
        """
        K-Means++ initialisation.
        z_batch: (N, latent_dim) tensor, N >= n_regions.
        """
        z = z_batch.detach().cpu().float()
        N = z.shape[0]
        assert N >= self.n_regions, (
            f'Need at least {self.n_regions} samples to seed, got {N}'
        )

        # First center: random
        centers = [z[torch.randint(N, (1,)).item()]]

        for _ in range(self.n_regions - 1):
            # Distance from every point to its nearest already-chosen center
            dist_sq = torch.stack(
                [((z - c.unsqueeze(0)) ** 2).sum(dim=1) for c in centers], dim=1
            ).min(dim=1).values                        # (N,)

            # Sample proportional to distance² — farther points more likely
            probs = dist_sq / (dist_sq.sum() + 1e-10)
            idx   = torch.multinomial(probs, 1).item()
            centers.append(z[idx])

        self.centroids = torch.stack(centers)          # (n_regions, latent_dim)

    # ------------------------------------------------------------------

    def _sq_dists(self, z_flat: torch.Tensor) -> torch.Tensor:
        """Squared L2 distances from z_flat (latent_dim,) to all centroids."""
        return ((self.centroids - z_flat.unsqueeze(0)) ** 2).sum(dim=1)  # (n_regions,)

    def query_and_update(self, z: torch.Tensor) -> int:
        """
        Assign z to nearest centroid and update it via EMA.
        z: (latent_dim,) or (1, latent_dim), detached.
        Returns region index.
        """
        z_flat  = z.detach().cpu().float().flatten()
        dists   = self._sq_dists(z_flat)
        region  = int(dists.argmin())
        self.centroids[region] = (
            self.momentum * self.centroids[region]
            + (1.0 - self.momentum) * z_flat
        )
        return region

    def query(self, z: torch.Tensor) -> int:
        """Assign z to nearest centroid without updating (inference mode)."""
        z_flat = z.detach().cpu().float().flatten()
        return int(self._sq_dists(z_flat).argmin())

    def distances(self, z: torch.Tensor) -> torch.Tensor:
        """Return (n_regions,) sqrt distances — useful for display."""
        z_flat = z.detach().cpu().float().flatten()
        return self._sq_dists(z_flat).sqrt()

In [8]:
pose_ae      = PoseAutoencoder().to(run_device)
detector     = RegionDetector(n_regions=n_regions, latent_dim=latent_dim, momentum=centroid_mom)
optimizer    = torch.optim.Adam(pose_ae.parameters(), lr=ae_lr)
criterion    = nn.MSELoss()

hand_tracker = HandTracker(model_path='hand_landmarker.task')
camera       = CameraWrapper(1)

In [9]:
from torch.utils.data import DataLoader, TensorDataset

# ── 1. Collect frames ──────────────────────────────────────────────────────────
warmup_raw = []
hand_tracker.reset()
t0 = time.time()

print(f'Collecting {warmup_frames} warmup frames.  Move your hand through all poses.')
print('Press SPACE to stop early.\n')

viz = HandVisualizer('Warmup', size=(400, 400))

while len(warmup_raw) < warmup_frames and not keyboard.is_pressed('space'):
    frame     = camera()
    hand_pose = hand_tracker(frame, int((time.time() - t0) * 1000))
    if hand_pose.size(0) != num_hands:
        continue
    offsets = to_bone_offsets(hand_pose)
    warmup_raw.append(offsets.cpu())
    viz.update(hand_pose)

viz.close()

assert len(warmup_raw) >= n_regions * 5, (
    f'Only {len(warmup_raw)} frames collected — need at least {n_regions * 5}. Re-run.'
)
print(f'Collected {len(warmup_raw)} frames.')

# ── 2. Batch-train AE on warmup data ──────────────────────────────────────────
warmup_tensor = torch.stack(warmup_raw)               # (N, 1, 21, 3)
warmup_dl     = DataLoader(TensorDataset(warmup_tensor), batch_size=32, shuffle=True)

pose_ae.train()
for ep in range(warmup_epochs):
    ep_loss = 0.0
    for (batch,) in warmup_dl:
        batch     = batch.to(run_device)
        recon, _  = pose_ae(batch)
        loss      = criterion(recon, batch.flatten(1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ep_loss  += loss.item()
    if (ep + 1) % 20 == 0:
        print(f'  Warmup epoch {ep+1:3d}/{warmup_epochs}  loss={ep_loss/len(warmup_dl):.6f}')

print('Warmup AE training done.')

# ── 3. Seed region detector ────────────────────────────────────────────────────
pose_ae.eval()
with torch.no_grad():
    z_warmup = pose_ae.encoder(warmup_tensor.to(run_device))   # (N, latent_dim)

detector.seed_from_batch(z_warmup.cpu())
print(f'Region detector seeded.  Centroid norms: {detector.centroids.norm(dim=1).tolist()}')

Press SPACE to stop early.

Collected 300 frames.
  Warmup epoch  20/80  loss=0.000333
  Warmup epoch  40/80  loss=0.000174
  Warmup epoch  60/80  loss=0.000117
  Warmup epoch  80/80  loss=0.000088
Warmup AE training done.
Region detector seeded.  Centroid norms: [0.5991930365562439, 0.36075305938720703, 0.6044529676437378, 0.31991302967071533]


In [10]:
replay_buf   = deque(maxlen=replay_size)
_WRITE_EVERY = 0.15   # seconds between display refreshes

hand_tracker.reset()
t0         = time.time()
last_write = 0.0
loss_val   = float('nan')
region_idx = -1

print("Live training running.  Press 'q' to stop.\n")

while not keyboard.is_pressed('q'):
    frame     = camera()
    hand_pose = hand_tracker(frame, int((time.time() - t0) * 1000))
    if hand_pose.size(0) != num_hands:
        continue

    offsets = to_bone_offsets(hand_pose)      # (1, 21, 3)
    replay_buf.append(offsets.cpu())

    # ── AE gradient step (experience replay) ──────────────────────────────────
    if len(replay_buf) >= replay_batch:
        mini     = torch.stack(random.sample(list(replay_buf), replay_batch)).to(run_device)
        recon, _ = pose_ae(mini)
        loss     = criterion(recon, mini.flatten(1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_val = loss.item()

    # ── Encode current frame (no grad) + region detection ─────────────────────
    with torch.no_grad():
        _, z = pose_ae(offsets.to(run_device))      # z: (1, latent_dim)

    region_idx = detector.query_and_update(z[0])

    # ── Display ───────────────────────────────────────────────────────────────
    now = time.time()
    if now - last_write >= _WRITE_EVERY:
        dists   = detector.distances(z[0])           # (n_regions,) sqrt distances
        max_d   = dists.max().item()

        label   = action_labels[region_idx] if 0 <= region_idx < len(action_labels) else '?'
        clear_output(wait=True)
        print(f'Region: [ {region_idx} ]  label: [ {label} ]   AE loss: {loss_val:.5f}\n')

        print('  Proximity to each region  (more filled = closer):')
        for k, d in enumerate(dists):
            d      = d.item()
            # Invert: closest centroid gets fullest bar
            prox   = 1.0 - d / (max_d + 1e-8)
            filled = int(prox * 20)
            bar    = '\u2588' * filled + '\u2591' * (20 - filled)
            arrow  = '  \u2190' if k == region_idx else ''
            lbl    = action_labels[k] if k < len(action_labels) else str(k)
            print(f'    {lbl}:  {bar}  d={d:.3f}{arrow}')

        last_write = now

print('\nStopped.')

Region: [ 1 ]  label: [ s ]   AE loss: 0.00001

  Proximity to each region  (more filled = closer):
    a:  ████░░░░░░░░░░░░░░░░  d=0.412
    s:  ████████████░░░░░░░░  d=0.188  ←
    d:  ███░░░░░░░░░░░░░░░░░  d=0.443
    w:  ░░░░░░░░░░░░░░░░░░░░  d=0.528


KeyboardInterrupt: 

In [ ]:
pose_ae.eval()
hand_tracker.reset()
t0       = time.time()
held_key = None

print("Inference — press 'q' to stop.\n")

with torch.no_grad():
    while not keyboard.is_pressed('q'):
        frame     = camera()
        hand_pose = hand_tracker(frame, int((time.time() - t0) * 1000))
        if hand_pose.size(0) != num_hands:
            continue

        offsets    = to_bone_offsets(hand_pose)
        _, z       = pose_ae(offsets.to(run_device))
        region_idx = detector.query(z[0])          # no centroid update

        label = action_labels[region_idx] if 0 <= region_idx < len(action_labels) else '?'

        if press_buttons and label != held_key:
            if held_key is not None:
                pyautogui.keyUp(held_key)
            pyautogui.keyDown(label)
            held_key = label

        dists  = detector.distances(z[0])
        max_d  = dists.max().item()
        clear_output(wait=True)
        print(f'Region: [ {region_idx} ]  →  action: [ {label} ]\n')
        for k, d in enumerate(dists):
            d      = d.item()
            prox   = 1.0 - d / (max_d + 1e-8)
            filled = int(prox * 20)
            bar    = '\u2588' * filled + '\u2591' * (20 - filled)
            arrow  = '  \u2190' if k == region_idx else ''
            lbl    = action_labels[k] if k < len(action_labels) else str(k)
            print(f'  {lbl}:  {bar}  d={d:.3f}{arrow}')

if held_key is not None:
    pyautogui.keyUp(held_key)

print('\nStopped.')

Region: [ 1 ]  →  action: [ s ]

  a:  ███░░░░░░░░░░░░░░░░░  d=0.447
  s:  ███████████░░░░░░░░░  d=0.250  ←
  d:  █░░░░░░░░░░░░░░░░░░░  d=0.518
  w:  ░░░░░░░░░░░░░░░░░░░░  d=0.555
